In [ ]:
pip install mysql-connector-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 49.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 40.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import re
from datetime import datetime
import mysql.connector

## Connecteur à la base de donnée

In [ ]:

def creationConnection():
  # Informations de connexion à la base de données MySQL
  db_host = 'madeshu.fr'
  db_user = 'superadmin'
  db_password = 'superadmin'
  db_name = 'MSPR2ETL_All'

  # Création de la connexion à la base de données MySQL
  conn = mysql.connector.connect(
      host=db_host,
      user=db_user,
      password=db_password,
      database=db_name
  )

  cursor = conn.cursor()
  return cursor, conn


## Extraction des données et formatage

### Fonctions utiles

In [ ]:
# Insertion bord
# INSERT INTO `DimBord` (`bord_id`, `nom_bord`) VALUES ('1', 'Extreme gauche'), ('2', 'Gauche'), ('3', 'Centre'), ('4', 'Droite'), ('5', 'Extreme droite'), ('6', 'Vert'), ('7', 'Non categorise');

In [ ]:
def extraire_date(texte):
    # Motif de recherche pour extraire la date
    motif = r"\d{2}/\d{2}/\d{4}"

    # Rechercher la date dans le texte en utilisant l'expression régulière
    resultats = re.search(motif, texte)

    if resultats:
        # Extraire la date trouvée
        date_str = resultats.group()

        # Convertir la date en objet de date
        date = datetime.strptime(date_str, "%d/%m/%Y").date()

        return date
    else:
        return None

LE PEN


In [ ]:
def extraire_mots_majuscules(texte):
  pattern = r'\b[A-Z]+\b(?![^(]*\))'
  mots_en_majuscules = re.findall(pattern, texte)

  resultat = ' '.join(mots_en_majuscules)
  return resultat

chemin_fichier = '/content/drive/MyDrive/1_EPSI/I1/MSPR2 - Big Data/data/Nos donnees/99 - Correspondance_candidat_bord_partie.xlsx'
dfcorrespondance = pd.read_excel(chemin_fichier, header=0)

# Récupère ou crée un partie à partir d'un nom
def CANDIDAT_getIdByName(cursor, nom_recherche, noregex = False):
    # Netoyage
    nom = extraire_mots_majuscules(nom_recherche)
    if(nom != "" and not noregex):
      nom_recherche = nom

    id_insered = None  # Variable pour stocker l'ID inséré

    resultats = dfcorrespondance[dfcorrespondance["Nom"] == nom_recherche]

    if resultats.empty:
      bord_associé = resultats["Bord"].iloc[0] if len(resultats) > 0 else None
      partie_associée = resultats["Parti"].iloc[0] if len(resultats) > 0 else None
    else:
      bord_associé = 'Inconnu'
      partie_associée = 'Inconnu'

    try:
        # Vérification de l'existence de la ligne dans la table (insensible à la casse)
        sql_check = "SELECT candidat_id FROM dim_candidat WHERE LOWER(candidat_nom) = LOWER(%s)"
        cursor.execute(sql_check, (nom_recherche,))
        result = cursor.fetchone()

        if result is not None:
            id_insered = result[0]
        else:
            # Insérer la ligne si elle n'existe pas déjà
            sql_insert = "INSERT INTO dim_candidat (`candidat_nom`,`bord_nom`,`partie_nom`) VALUES (%s, %s, %s)"
            values = (nom_recherche,bord_associé,partie_associée)
            cursor.execute(sql_insert, values)
            conn.commit()
            id_insered = cursor.lastrowid

        #print("Insertion des données terminée.")

    except mysql.connector.Error as error:
        print("Erreur lors de l'insertion des données :", error)
        raise Exception("Erroeur lors de l'insertion du partie " + "nom_recherche")

    return id_insered

### Initialisation de l'environnement

In [ ]:
#def DELETE_ALL(cursor):
#  cursor.execute("DELETE FROM Participation_Election")
#  cursor.execute("DELETE FROM Partie")
#  cursor.execute("DELETE FROM Elections")
#try:
#  cursor, conn = creationConnection()
#
#  DELETE_ALL(cursor)
#  conn.commit()
#finally :
#  # Fermeture de la connexion à la base de données MySQL
#  cursor.close()
#  conn.close()*/

### Transformation des données

In [ ]:
## Dans la bdd penser à mettre le departement_code en varchar(10) sinon la corse ne passe pas
## Dans la bdd penser à supprimer le champ circonscription
## Dans la bdd penser à renomer dans Participation_Election id par id_Election et resultat par nombre_vote

# Connection à la base de donnée
cursor, conn = creationConnection()

# Chemin vers le dossier contenant les fichiers Excel
dossier_excel = '/content/drive/MyDrive/1_EPSI/I1/MSPR2 - Big Data/data/Elections présidentielles/1965 - 2022 Talend'


# Itération sur les fichiers Excel du dossier

try :
  for fichier in os.listdir(dossier_excel):
      if fichier.endswith('.xlsx') or fichier.endswith('.xls'):
          chemin_fichier = os.path.join(dossier_excel, fichier)
          nom_fichier_sans_extension = os.path.splitext(fichier)[0]

          # Vérification fichier
          if (fichier == "Presidentielle_2017_Resultats_Tour_1_c.xls" or fichier == "resultats-par-niveau-cirlg-t1-france-entiere.xlsx"):
            continue

          # Lecture du fichier Excel dans un dataframe
          dfDate = pd.read_excel(chemin_fichier, header=None, sheet_name=nom_fichier_sans_extension)
          dfdata = pd.read_excel(chemin_fichier, header=4, sheet_name=nom_fichier_sans_extension)

          # Affichage
          print("Traitement du fichie" + fichier)

          # Vérification de l'année
          election_date = extraire_date(str(dfDate.iat[1, 0]))
          if (election_date.year >= 2017):
            continue

          ### TRANSFORM & LOAD

          #groupe_departement = dfdata.groupby(["Code département", "département"]).agg("sum").reset_index()
          #print(groupe_departement.head().to_string(index=False))
          print(dfdata.head().to_string(index=False))

          ## FactElection
          nombre_vote_total = str(dfdata['Votants'].sum())
          nombre_abstention_total = str(dfdata['Blancs et nuls'].sum())
          election_date = extraire_date(str(dfDate.iat[1, 0]))

          query = "INSERT INTO fact_election (`nombre_abstention_total`, `nombre_vote_total`, `election_date`) VALUES (%s, %s, %s)"
          valeurs = (nombre_abstention_total, nombre_vote_total, election_date)
          cursor.execute(query, valeurs)
          conn.commit()

          election_id = cursor.lastrowid

          # Les 8 première colonnes sont dédiées au stats
          # Nous récupérons donc les colonnes suivantes pour avoir le nombre de vote par candidat par département
          for candidat_column in dfdata.columns[7:]:
            # Initialiser les listes pour les colonnes nom et vote
            # Parcourir les colonnes du dataframe et extraire les valeurs de nom et de vote

            ## DimCandidat
            candidat_id = CANDIDAT_getIdByName(cursor, candidat_column) #DimCandidat

            ## FactDetailElection
            # Préparation des données
            nombre_vote = int(dfdata[candidat_column].sum())
            election_id = election_id
            candidat_id = candidat_id

            # Préparation de l'insertion des données
            query = "INSERT INTO fact_resultat_candidat (nombre_vote, election_id, candidat_id) VALUES (%s, %s, %s);"
            values = (nombre_vote, election_id, candidat_id)
            cursor.execute(query, values)

  # Confirmation de la requête
  conn.commit()

#except:

#  conn.rollback()


finally :
  # Fermeture de la connexion à la base de données MySQL
  conn.rollback()
  cursor.close()
  conn.close()
  print("ETL election terminé")



Traitement du fichiecdsp_presi1965t1_circ.xls
 Code département département  circonscription  Inscrits  Votants  Exprimés  Blancs et nuls  MITTERRAND (CIR)  LECANUET (MRP)  DE GAULLE (UNR)  MARCILHACY (DVD)  TIXIER-VIGNANCOUR (EXD)  BARBU (DIV)
                1         AIN                1     68818    54262     53769             493             16677           11669            21166              1017                     2591          649
                1         AIN                2     70186    57221     56780             441             18247            7936            26645               924                     2366          662
                1         AIN                3     67492    55503     55006             497             15494           10811            23435              1065                     3360          841
                2       AISNE                1     52618    46064     45564             500             12999            7242            22647               5

### Traitement des données 2017

In [ ]:
## Dans la bdd penser à mettre le departement_code en varchar(10) sinon la corse ne passe pas
## Dans la bdd penser à supprimer le champ circonscription
## Dans la bdd penser à renomer dans Participation_Election id par id_Election et resultat par nombre_vote

# Connection à la base de donnée
cursor, conn = creationConnection()

# Chemin vers le dossier contenant les fichiers Excel
dossier_excel = '/content/drive/MyDrive/1_EPSI/I1/MSPR2 - Big Data/data/Elections présidentielles/1965 - 2022 Talend'


# Itération sur les fichiers Excel du dossier

try :
  for fichier in os.listdir(dossier_excel):
      if fichier.endswith('.xlsx') or fichier.endswith('.xls'):
          chemin_fichier = os.path.join(dossier_excel, fichier)
          nom_fichier_sans_extension = os.path.splitext(fichier)[0]

          # Lecture du fichier Excel dans un dataframe
          dfDate = pd.read_excel(chemin_fichier, header=None)
          dfdata = pd.read_excel(chemin_fichier, header=3)

          # Affichage
          print("Traitement du fichie" + fichier)

          # Vérification de l'année
          if (fichier == "Presidentielle_2017_Resultats_Tour_1_c.xls"):
            election_date = datetime.strptime('2017-04-23', "%Y-%m-%d")
          else:
            continue

          ### TRANSFORM & LOAD

          #groupe_departement = dfdata.groupby(["Code département", "département"]).agg("sum").reset_index()
          #print(groupe_departement.head().to_string(index=False))
          print(dfdata.head().to_string(index=False))

          ## FactElection
          nombre_vote_total = str(dfdata['Votants'].sum())
          nombre_abstention_total = str(dfdata['Blancs'].sum() + dfdata['Nuls'].sum())
          election_date = election_date

          query = "INSERT INTO fact_election (`nombre_abstention_total`, `nombre_vote_total`, `election_date`) VALUES (%s, %s, %s)"
          valeurs = (nombre_abstention_total, nombre_vote_total, election_date)
          cursor.execute(query, valeurs)
          conn.commit()

          election_id = cursor.lastrowid


          nom_column = []
          nombre_vote = []
          for index, row in dfdata.iterrows():
            for candidat_column in dfdata.columns[17:]:
              if("Nom" in candidat_column ):
                nom_column.append(row[candidat_column])
              elif "Voix" in candidat_column and "Ins" not in candidat_column and "Exp" not in candidat_column:
                nombre_vote.append(row[candidat_column])

          dfvote = pd.DataFrame({'nom': nom_column, 'vote': nombre_vote}).groupby(["nom"]).agg("sum").reset_index()
          print(dfvote.head().to_string(index=False))

          for index, row in dfvote.iterrows():
            ## DimCandidat
            candidat_id = CANDIDAT_getIdByName(cursor, row['nom'], True) #DimCandidat

            ## FactDetailElection
            # Préparation des données
            nombre_vote = row['vote']
            election_id = election_id
            candidat_id = candidat_id

            # Préparation de l'insertion des données
            query = "INSERT INTO fact_resultat_candidat (nombre_vote, election_id, candidat_id) VALUES (%s, %s, %s);"
            values = (nombre_vote, election_id, candidat_id)
            cursor.execute(query, values)


  # Confirmation de la requête
  conn.commit()

#except:

#  conn.rollback()


finally :
  # Fermeture de la connexion à la base de données MySQL
  conn.rollback()
  cursor.close()
  conn.close()
  print("ETL election terminé")

Traitement du fichiecdsp_presi1965t1_circ.xls
Traitement du fichiecdsp_presi1969t1_circ.xls
Traitement du fichiecdsp_presi1974t1_circ.xls
Traitement du fichiecdsp_presi1981t1_circ.xls
Traitement du fichiecdsp_presi1988t1_circ.xls
Traitement du fichiecdsp_presi1995t1_circ.xls
Traitement du fichiecdsp_presi2002t1_circ.xls
Traitement du fichiecdsp_presi2012t1_circ.xlsx
Traitement du fichiecdsp_presi2007t1_circ.xlsx
Traitement du fichiePresidentielle_2017_Resultats_Tour_1_c.xls
Code du département Libellé du département  Code de la circonscription Libellé de la circonscription  Inscrits  Abstentions  % Abs/Ins  Votants  % Vot/Ins  Blancs  % Blancs/Ins  % Blancs/Vot  Nuls  % Nuls/Ins  % Nuls/Vot  Exprimés  % Exp/Ins  % Exp/Vot  N°Panneau Sexe    Nom   Prénom  Voix  % Voix/Ins  % Voix/Exp  N°Panneau.1 Sexe.1  Nom.1 Prénom.1  Voix.1  % Voix/Ins.1  % Voix/Exp.1  N°Panneau.2 Sexe.2  Nom.2 Prénom.2  Voix.2  % Voix/Ins.2  % Voix/Exp.2  N°Panneau.3 Sexe.3     Nom.3 Prénom.3  Voix.3  % Voix/Ins.3  

Traitement donnée 2022

In [ ]:
## Dans la bdd penser à mettre le departement_code en varchar(10) sinon la corse ne passe pas
## Dans la bdd penser à supprimer le champ circonscription
## Dans la bdd penser à renomer dans Participation_Election id par id_Election et resultat par nombre_vote

# Connection à la base de donnée
cursor, conn = creationConnection()

# Chemin vers le dossier contenant les fichiers Excel
dossier_excel = '/content/drive/MyDrive/1_EPSI/I1/MSPR2 - Big Data/data/Elections présidentielles/1965 - 2022 Talend'


# Itération sur les fichiers Excel du dossier

try :
  for fichier in os.listdir(dossier_excel):
      if fichier.endswith('.xlsx') or fichier.endswith('.xls'):
          chemin_fichier = os.path.join(dossier_excel, fichier)
          nom_fichier_sans_extension = os.path.splitext(fichier)[0]

          # Lecture du fichier Excel dans un dataframe
          dfDate = pd.read_excel(chemin_fichier, header=None)
          dfdata = pd.read_excel(chemin_fichier, header=0)

          # Affichage
          print("Traitement du fichie" + fichier)

          # Vérification de l'année
          if (fichier == "resultats-par-niveau-cirlg-t1-france-entiere.xlsx"):
            election_date = datetime.strptime('2022-04-24', "%Y-%m-%d")
          else:
            continue

          ### TRANSFORM & LOAD

          #groupe_departement = dfdata.groupby(["Code département", "département"]).agg("sum").reset_index()
          #print(groupe_departement.head().to_string(index=False))
          print(dfdata.head().to_string(index=False))

          ## FactElection
          nombre_vote_total = str(dfdata['Votants'].sum())
          nombre_abstention_total = str(dfdata['Blancs'].sum() + dfdata['Nuls'].sum())
          election_date = election_date

          query = "INSERT INTO fact_election (`nombre_abstention_total`, `nombre_vote_total`, `election_date`) VALUES (%s, %s, %s)"
          valeurs = (nombre_abstention_total, nombre_vote_total, election_date)
          cursor.execute(query, valeurs)
          conn.commit()

          election_id = cursor.lastrowid


          nom_column = []
          nombre_vote = []
          itteration = 0
          for candidat_column in dfdata.columns[19:]:
            itteration = itteration + 1
            if(itteration == 3 ):
              nom_column.append(dfdata[candidat_column][0])
            elif itteration == 5:
              nombre_vote.append(dfdata[candidat_column].sum())
            elif itteration == 7:
              itteration = 0

          dfvote = pd.DataFrame({'nom': nom_column, 'vote': nombre_vote}).reset_index()
          print(dfvote.head().to_string(index=False))

          for index, row in dfvote.iterrows():
            ## DimCandidat
            candidat_id = CANDIDAT_getIdByName(cursor, row['nom'], True) #DimCandidat

            ## FactDetailElection
            # Préparation des données
            nombre_vote = row['vote']
            election_id = election_id
            candidat_id = candidat_id

            # Préparation de l'insertion des données
            query = "INSERT INTO fact_resultat_candidat (nombre_vote, election_id, candidat_id) VALUES (%s, %s, %s);"
            values = (nombre_vote, election_id, candidat_id)
            cursor.execute(query, values)


  # Confirmation de la requête
  conn.commit()

#except:

#  conn.rollback()


finally :
  # Fermeture de la connexion à la base de données MySQL
  conn.rollback()
  cursor.close()
  conn.close()
  print("ETL election terminé")

Traitement du fichiecdsp_presi1965t1_circ.xls
Traitement du fichiecdsp_presi1969t1_circ.xls
Traitement du fichiecdsp_presi1974t1_circ.xls
Traitement du fichiecdsp_presi1981t1_circ.xls
Traitement du fichiecdsp_presi1988t1_circ.xls
Traitement du fichiecdsp_presi1995t1_circ.xls
Traitement du fichiecdsp_presi2002t1_circ.xls
Traitement du fichiecdsp_presi2012t1_circ.xlsx
Traitement du fichiecdsp_presi2007t1_circ.xlsx
Traitement du fichiePresidentielle_2017_Resultats_Tour_1_c.xls
Traitement du fichieresultats-par-niveau-cirlg-t1-france-entiere.xlsx
Code du département Libellé du département  Code de la circonscription Libellé de la circonscription Etat saisie  Inscrits  Abstentions  % Abs/Ins  Votants  % Vot/Ins  Blancs  % Blancs/Ins  % Blancs/Vot  Nuls  % Nuls/Ins  % Nuls/Vot  Exprimés  % Exp/Ins  % Exp/Vot  N°Panneau Sexe     Nom   Prénom  Voix  % Voix/Ins  % Voix/Exp  Unnamed: 26 Unnamed: 27 Unnamed: 28 Unnamed: 29  Unnamed: 30  Unnamed: 31  Unnamed: 32  Unnamed: 33 Unnamed: 34 Unnamed: 3

### Prompt GPT pour les parties et les bords :
Nous n'avons pas trouvé de liste exaustive pour trouver le bord et la partie des candidtas.

Nous utilisons l'outil ChatGPT pour trouver les informations manquante.
Voici comment procéder pour trouver les bord et partie des candidats sans données.

Récupérer la liste des candidats sans Bord et sans Parti :


```
SELECT candidat_nom FROM `dim_candidat` WHERE bord_nom = 'Inconnu' or partie_nom = 'Inconnu'
```

Ensuite avec les noms, donnez le prompt suivant à chatgpt :

```
Avec la liste suivante d'homme politique,
écrit leur bord associé selon la liste suivante (Extreme gauche, Gauche, Centre, Droit, Extreme droit, Vert, Non categorise) ,
écrit aussi leur partie, si ils n'ont pas de partie, marque "Inconnu".
Formate les lignes de cette façon :
UPDATE dim_candidat SET bord_nom="BORD_ASSOCIÉ", partie_nom="PARTIE_ASSOCIÉ" WHERE candidat_nom = "NOM_DU_POLITIQUE" :
[Collez la liste des candidats ici]
```

Voici un exemple de prompt pour corriger les bords et les parties :


```
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "MITTERRAND";
UPDATE dim_candidat SET bord_nom="Centre", partie_nom="Parti Républicain" WHERE candidat_nom = "LECANUET";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Rassemblement du Peuple Français" WHERE candidat_nom = "DE GAULLE";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "MARCILHACY";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "TIXIER VIGNANCOUR";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "BARBU";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Ligue Communiste Révolutionnaire" WHERE candidat_nom = "KRIVINE";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "ROCARD";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "DUCLOS";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "DEFFERRE";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Union pour la Nouvelle République" WHERE candidat_nom = "POMPIDOU";
UPDATE dim_candidat SET bord_nom="Centre", partie_nom="Centre Démocratie et Progrès" WHERE candidat_nom = "POHER";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "DUCATEL";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Lutte Ouvrière" WHERE candidat_nom = "LAGUILLER";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "MULLER";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "DUMONT";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Union pour la Démocratie Française" WHERE candidat_nom = "GISCARD D ESTAING";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Rassemblement pour la République" WHERE candidat_nom = "CHABAN DELMAS";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "RENOUVIN";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "ROYER";
UPDATE dim_candidat SET bord_nom="Extrême droite", partie_nom="Front National" WHERE candidat_nom = "LE PEN";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "HERAUD";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "SEBAG";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "BOUCHARDEAU";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "MARCHAIS";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "CREPEAU";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Génération Écologie" WHERE candidat_nom = "LALONDE";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Rassemblement pour la République" WHERE candidat_nom = "CHIRAC";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Rassemblement pour la République" WHERE candidat_nom = "DEBRE";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Rassemblement pour la République" WHERE candidat_nom = "GARAUD";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "BOUSSEL";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "LAJOINIE";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "JUQUIN";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Les Verts" WHERE candidat_nom = "WAECHTER";
UPDATE dim_candidat SET bord_nom="Centre-droit", partie_nom="Union pour la Démocratie Française" WHERE candidat_nom = "BARRE";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "HUE";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "JOSPIN";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Les Verts" WHERE candidat_nom = "VOYNET";
UPDATE dim_candidat SET bord_nom="Centre-droit", partie_nom="Rassemblement pour la République, puis Démocratie Libérale" WHERE candidat_nom = "BALLADUR";
UPDATE dim_candidat SET bord_nom="Droite conservatrice", partie_nom="Mouvement pour la France" WHERE candidat_nom = "DE VILLIERS";
UPDATE dim_candidat SET bord_nom="Extrême droite", partie_nom="Solidarité et Progrès" WHERE candidat_nom = "CHEMINADE";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Ligue Communiste Révolutionnaire" WHERE candidat_nom = "GLUCKSTEIN";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Nouveau Parti Anticapitaliste" WHERE candidat_nom = "BESANCENOT";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "TAUBIRA";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Mouvement des Citoyens" WHERE candidat_nom = "CHEVENEMENT";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Les Verts, puis Europe Écologie Les Verts" WHERE candidat_nom = "MAMERE";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Les Verts, puis Europe Écologie Les Verts" WHERE candidat_nom = "LEPAGE";
UPDATE dim_candidat SET bord_nom="Centre", partie_nom="Mouvement Démocrate" WHERE candidat_nom = "BAYROU";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Démocratie Libérale" WHERE candidat_nom = "MADELIN";
UPDATE dim_candidat SET bord_nom="Droite conservatrice", partie_nom="Parti Chrétien-Démocrate" WHERE candidat_nom = "BOUTIN";
UPDATE dim_candidat SET bord_nom="Extrême droite", partie_nom="Front National, puis Mouvement National Républicain" WHERE candidat_nom = "MEGRET";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "SAINT JOSSE";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Inconnu" WHERE candidat_nom = "SCHIVARDI";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Confédération Paysanne, puis Les Verts" WHERE candidat_nom = "BOVE";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti Communiste Français" WHERE candidat_nom = "BUFFET";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "ROYAL";
UPDATE dim_candidat SET bord_nom="Droit", partie_nom="Union pour un Mouvement Populaire, puis Les Républicains" WHERE candidat_nom = "SARKOZY";
UPDATE dim_candidat SET bord_nom="Droite conservatrice", partie_nom="Mouvement pour la France" WHERE candidat_nom = "DE VILLIERS";
UPDATE dim_candidat SET bord_nom="Droite conservatrice", partie_nom="Chasse, Pêche, Nature et Traditions" WHERE candidat_nom = "NIHOUS";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Lutte Ouvrière" WHERE candidat_nom = "ARTHAUD";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Nouveau Parti Anticapitaliste" WHERE candidat_nom = "POUTOU";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="La France insoumise" WHERE candidat_nom = "MELENCHON";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "HOLLANDE";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Europe Écologie Les Verts" WHERE candidat_nom = "JOLY";
UPDATE dim_candidat SET bord_nom="Droite conservatrice", partie_nom="Debout la France" WHERE candidat_nom = "DUPONT AIGNAN";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Lutte Ouvrière" WHERE candidat_nom = "ARTHAUD";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti communiste français" WHERE candidat_nom = "ROUSSEL";
UPDATE dim_candidat SET bord_nom="Centre", partie_nom="En Marche" WHERE candidat_nom = "MACRON";
UPDATE dim_candidat SET bord_nom="Centre", partie_nom="Résistons !" WHERE candidat_nom = "LASSALLE";
UPDATE dim_candidat SET bord_nom="Extrême droite", partie_nom="Rassemblement National" WHERE candidat_nom = "LE PEN";
UPDATE dim_candidat SET bord_nom="Droite", partie_nom="Debout la France" WHERE candidat_nom = "ZEMMOUR";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="La France Insoumise" WHERE candidat_nom = "MÉLENCHON";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "HIDALGO";
UPDATE dim_candidat SET bord_nom="Vert", partie_nom="Europe Écologie Les Verts" WHERE candidat_nom = "JADOT";
UPDATE dim_candidat SET bord_nom="Droite", partie_nom="Les Républicains" WHERE candidat_nom = "PÉCRESSE";
UPDATE dim_candidat SET bord_nom="Extrême gauche", partie_nom="Nouveau Parti Anticapitaliste" WHERE candidat_nom = "POUTOU";
UPDATE dim_candidat SET bord_nom="Droite", partie_nom="Debout la France" WHERE candidat_nom = "DUPONT-AIGNAN";
UPDATE dim_candidat SET bord_nom="Non catégorisé", partie_nom="Union Populaire Républicaine" WHERE candidat_nom = "ASSELINEAU";
UPDATE dim_candidat SET bord_nom="Extrême droite", partie_nom="Solidarité et Progrès" WHERE candidat_nom = "CHEMINADE";
UPDATE dim_candidat SET bord_nom="Droite", partie_nom="Les Républicains" WHERE candidat_nom = "FILLON";
UPDATE dim_candidat SET bord_nom="Gauche", partie_nom="Parti socialiste" WHERE candidat_nom = "HAMON";
```

